# UCMCTrack en tiempo real — con calibración AUTOMÁTICA de la cámara

**Qué hace.** UCMCTrack (AAAI 2024) no asocia en la imagen sino en el **suelo**: cada detección se proyecta al plano del suelo (metros) con un modelo de cámara, y ahí un Kalman de velocidad constante predice y asocia con la distancia de Mahalanobis (MMD). En metros, un auto a 60 km/h se mueve igual cerca o lejos de la cámara; en píxeles, no.

**El problema.** Para proyectar al suelo necesita la cámara: inclinación (**pitch**), campo de visión (**hfov**) y **altura**. El original la pide **a mano** (`cam_para.txt`) y la supone **fija**. Desde un dron no es fija: el gimbal cambia de inclinación y el dron sube y baja.

### De manual a automático

| | MANUAL (UCMCTrack original / v12) | AUTOMÁTICO (v13, aquí en línea) |
|---|---|---|
| **Pitch** | A ojo, uno para todo el video | **Se mide** cada 0.5 s descomponiendo la `H` del suelo (celda 3.2) |
| **Altura** | A ojo, o una vez con los vehículos | **Forma** con la `H` (cuánto sube o baja) + **escala** con el tamaño de los vehículos (celda 3.2) |
| **Cámara en el tiempo** | Fija | El mapa imagen↔suelo se rehace cuando cambian el pitch o la altura |
| **CMC** | Una sola cámara | Sale al frame anterior con la cámara anterior y vuelve al suelo con la nueva (celda 3.3) |
| **hfov** | Dato de la cámara | **Sigue siendo un dato de la cámara** (ficha técnica, 82° por defecto) |
| **Cuándo se calcula** | Antes de empezar | v13: una pasada previa por el video completo. **Aquí: frame a frame, solo con el pasado** |

> **El hfov NO se estima automáticamente.** Se probó estimarlo con el tamaño de los vehículos (un auto debe medir lo mismo en el suelo cerca y lejos). Solo fue fiable en vigia_8 (≈64°) y vigia_9 (≈55°); en vigia_1 y vigia_10 no converge y en vigia_11 es ruido, así que no se incluyó. Hay que ponerlo en `CamaraUCMC(hfov_deg=...)` con el dato del dron: el pitch estimado depende de él (en vigia_8: 52° con hfov 60, 63° con 82, 71° con 95).

**Pipeline de un frame** (`TrackerUCMC.actualizar`):

| Paso | Qué hace | Celda |
|---|---|---|
| 1. Escala | frame original → resolución de trabajo (960 px) | 2.1 |
| 2. Estabilización | `H` = cómo se movió la cámara desde el frame anterior | 2.2 |
| 3. Entrada | detecciones a trabajo, sin cajas gigantes, + propuestas | 2.3 |
| 4. Calibración | pitch y altura de este frame → mapa imagen↔suelo | 3.1 / 3.2 |
| 5. UCMC | al suelo, predicción + CMC con `H`, 3 asociaciones MMD, nacer/morir | 3.3 |
| 6. Libro | salida de UCMC → `Track` con historial estabilizado | 2.5 |
| 7. ¿Se mueve? | igual que en todos los trackers | 4.1 |
| 8. Salida | tracks confirmados en píxeles originales | 4.2 |

**Entra en cada frame:**
- `frame`: imagen BGR a resolución **original** (el tracker la reduce a `work_width` = 960 px).
- `detecciones`: `Nx6 [x1, y1, x2, y2, conf, clase]` en píxeles del frame **original** (`clase` = id entero ≥ 0).

**Sale de cada frame:** lista de `TrackSalida` (solo tracks **confirmados** con detección en este frame): `id`, `caja` (px originales), `conf`, `clase` / `nombre_clase`, `en_movimiento`, `desplazamiento_px`, `paralaje`. Para un benchmark MOT: `como_array(tracks)` → `Nx7 [x1, y1, x2, y2, id, conf, clase]`.

**Llevarlo al otro entorno:** allí ya hay detector, así que solo hacen falta las PARTES 1–4 (o los `.py` de `VD - Trackers To Real Implementation/`, que tienen exactamente este mismo código). La PARTE 5 es el ejemplo de bucle: la celda **5.1** es la única que se sustituye por el detector real.

**Supuestos:** suelo plano, sin alabeo (el gimbal mantiene el roll ≈ 0) y que el dron se **desplace** de vez en cuando (en hover la `H` no dice nada de la inclinación: se conserva la última medida).

In [ ]:
# ======================================================================
# IMPORTACIONES
# Qué hace : Carga las librerías (el notebook no importa nada de VD - Trackers To Real Implementation/: todo el código está aquí)
# ¿Se toca? : No
# ======================================================================
import time
from collections import defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path

from scipy.optimize import linear_sum_assignment

import cv2
import numpy as np
import pandas as pd
from IPython.display import Video, display

## PARTE 1 — Configuración (lo único que se ajusta)
Todos los parámetros están agrupados por etiqueta: `[CAMARA]`, `[ENTRADA]`, `[ASOCIACION]`, `[CLASE]`, `[MOVIMIENTO]`, `[PARALAJE]`, `[PROPUESTAS]`.

- **1.1 `Config`**: lo común a los 4 trackers. Los marcados "igual para TODOS" deben ser iguales al comparar trackers.
- **Benchmark MOT:** `use_proposals=False` (las propuestas son cajas sin clase y el benchmark las contaría como falsos positivos).
- Las distancias están en píxeles de la resolución de **trabajo** (960 px), no de la original.
- **1.2 `ConfigUCMC` + `CamaraUCMC`**:
  - `[UCMC]`: valores del `demo.py` del repo original. **No** se ajustan con los videos de evaluación.
  - `[CALIBRACION EN LINEA]`: `ucmc_calibrar=False` = modo **MANUAL** (pitch de respaldo y altura fijos todo el video).
  - **`CamaraUCMC` sí se toca, por cámara/video:**
    - `hfov_deg`: dato de la ficha técnica del dron (82° por defecto). **No se estima.**
    - `pitch_respaldo_deg`: estimación a ojo; solo se usa hasta la 1ª medida de la `H` (o siempre si el dron no se desplaza). También decide el **punto de apoyo**: centro de la caja si < 30° (cenital), base si no.
    - `altura_m=None`: automática. Con un número (p. ej. la telemetría), queda fija.

In [ ]:
# ======================================================================
# 1.1 · Config
# Qué hace : Parámetros comunes a todos los trackers (los del tracker están en la celda 1.2)
# ¿Se toca? : SÍ — pero los marcados "igual para TODOS" deben ser iguales al comparar trackers
# ======================================================================
@dataclass
class Config:
    work_width: int = 960          # ancho al que se estima la H y se hace el tracking (el frame se reduce a este ancho)
    # [CAMARA] movimiento global de la cámara (homografía)
    min_inliers: int = 30          # por debajo, la homografía se considera poco fiable
    ransac_px: float = 1.5         # tolerancia RANSAC: más baja = se ajusta más al plano del suelo
    grid: tuple = (4, 3)           # (cols, filas): reparte los puntos por toda la imagen (evita que domine una zona)
    # [ENTRADA] qué detecciones llegan al tracker (igual para TODOS)
    max_box_area_frac: float = 0.03  # se descartan detecciones con caja mayor a esta fracción del frame
    use_proposals: bool = True     # añadir "propuestas": cajas SIN clase donde algo se mueve y el detector no ve nada
    # [ASOCIACION] umbrales comunes (igual para TODOS: ByteTrack, BoT-SORT y UCMC usan los mismos)
    conf_high: float = 0.35        # detecciones "fuertes" (1ª asociación, crean tracks nuevos)
    conf_low: float = 0.10         # detecciones "débiles" (2ª asociación, estilo ByteTrack)
    max_miss_sec: float = 1.0      # segundos que un track sobrevive sin detección
    min_hits: int = 4              # detecciones para confirmar un track (antes no sale)
    # [CLASE] evita que una detección suelta de baja confianza cambie la etiqueta de un track
    cls_min_conf: float = 0.40     # solo las detecciones con esta confianza votan por una clase
    cls_min_hits: int = 3          # votos mínimos para que el track adopte una clase; si no, queda "desconocido" (-1)
    # [MOVIMIENTO] cuándo un track cuenta como 'en movimiento' (igual para TODOS)
    window_sec: float = 1.0        # ventana temporal para medir el desplazamiento neto
    min_move_px: float = 6.0       # desplazamiento mínimo (en la ventana) para considerar movimiento
    rel_move: float = 0.30         # ...y además >= rel_move * tamaño del objeto (sqrt(ancho*alto)); frena cajas grandes inestables
    adaptive: bool = False         # umbral adaptativo con la población de tracks (solo si la mayoría es estática)
    max_adaptive_px: float = 12.0  # tope del umbral adaptativo
    mad_k: float = 4.0             # umbral adaptativo = mediana + k*MAD de la población de tracks
    on_frames: int = 3             # evaluaciones seguidas "moviéndose" para activar el estado
    off_frames: int = 10           # evaluaciones seguidas "quieto" para desactivarlo
    min_straightness: float = 0.60 # desplazamiento neto / recorrido en la ventana; un objeto real avanza en línea, el ruido deambula
    max_unknown_aspect: float = 6.0  # un track SIN clase más alargado que esto es una estructura (cable, borde), no un objeto
    # [PARALAJE] lo que sobresale del suelo se desplaza respecto a él cuando el dron se traslada
    parallax_filter: bool = True
    parallax_angle_deg: float = 30.0   # el residuo debe ser paralelo al flujo del suelo en ese punto
    parallax_k_min: float = -1.5       # k<0: algo fijo en la IMAGEN (cable que resbala, borde)
    parallax_k_max: float = 3.0        # k>0: paralaje (poste, árbol); un vehículo real tiene k >> 3
    parallax_min_fit: float = 0.6      # R² mínimo del ajuste  residuo(t) ≈ k·flujo_suelo_acumulado(t)
    parallax_min_ground_px: float = 3.0  # flujo NETO del suelo mínimo en la ventana (en hover no se decide)
    parallax_skip: tuple = ("car", "truck", "bus", "motorcycle", "small vehicle", "large vehicle")  # clases que NO se filtran
    # [PROPUESTAS] cambios de imagen sin clase: objetos que el detector no conoce (solo si use_proposals)
    proposal_gap_sec: float = 0.15 # separación temporal entre los frames que se restan
    diff_thr: int = 25             # diferencia de gris mínima para contar un píxel como "cambió"
    min_area: int = 25
    max_area: int = 6000
    dilate: int = 3
    border: int = 15               # se ignora el borde (ahí la imagen alineada no tiene datos)
    max_aspect: float = 5.0
    max_residual: float = 3.5      # diferencia media tras alinear; si se supera, la H falló y no se generan propuestas
    proposal_conf: float = 0.5     # confianza que se le da a una propuesta
    proposal_memory_sec: float = 0.5   # segundos que se recuerdan las detecciones (para no proponer sobre ellas)
    proposal_overlap: float = 0.2      # se descarta la propuesta si esta fracción de su área cae sobre una detección recordada
    max_coherence: float = 0.80    # estructura de una sola orientación (cables, líneas, bordillos) -> no es un objeto


def warp_pts(H, pts):
    """Aplica la homografía H a una lista de puntos (N x 2)."""
    pts = np.asarray(pts, np.float32).reshape(-1, 2)
    if len(pts) == 0:
        return pts
    return cv2.perspectiveTransform(pts.reshape(-1, 1, 2), H).reshape(-1, 2)


def iou_pad(a, b, pad):
    """IoU entre cajas (N x 4 contra M x 4) infladas `pad` px por lado (ayuda con objetos de pocos píxeles)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    a = a + np.array([-pad, -pad, pad, pad]); b = b + np.array([-pad, -pad, pad, pad])
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)

In [ ]:
# ======================================================================
# 1.2 · ConfigUCMC + CamaraUCMC
# Qué hace : Parámetros del algoritmo (ConfigUCMC) y datos de la cámara de cada dron/video (CamaraUCMC)
# ¿Se toca? : CamaraUCMC SÍ (por cámara). [UCMC] no: valores del repo original, fijados antes de evaluar
# ======================================================================
@dataclass
class ConfigUCMC(Config):
    # [UCMC] valores de demo.py del repo original (tráfico vehicular). NO ajustarlos con los videos de evaluación.
    # Umbrales de confianza y ventana de pérdida: los comunes (conf_high, conf_low, max_miss_sec).
    ucmc_a: float = 100.0          # umbral de asignación (costo MMD = Mahalanobis² + log|S|)
    ucmc_wx: float = 5.0           # ruido de proceso en x (aceleración, (m/s²)²)
    ucmc_wy: float = 5.0           # ruido de proceso en y
    ucmc_vmax: float = 10.0        # velocidad máxima esperada al nacer el track (m/s): incertidumbre inicial
    ucmc_uv_ratio: float = 0.05    # error de la detección en píxeles = 5 % de la altura de la caja (con topes)
    ucmc_punto: str = "auto"       # punto de apoyo en el suelo: "base" (original), "centro" o "auto" (centro si pitch_respaldo < 30°)
    # [ALTURA] escala en metros a partir de los vehículos
    ucmc_largo_vehiculo_m: float = 4.5    # tamaño real supuesto del lado mayor de un vehículo
    ucmc_altura_defecto_m: float = 100.0  # hasta que haya vehículos para estimarla
    # [CALIBRACION EN LINEA] (v13, hecha causal)
    ucmc_calibrar: bool = True            # False = modo MANUAL: pitch_respaldo_deg y altura fijos todo el tiempo
    ucmc_calib_ventana_s: float = 1.0     # separación de los dos frames cuya H se descompone
    ucmc_calib_min_t: float = 0.03        # desplazamiento mínimo del dron en la ventana (relativo a la distancia al suelo)
    ucmc_calib_max_roll: float = 25.0     # soluciones con más alabeo que esto se descartan (el gimbal mantiene el roll ~0)
    ucmc_calib_memoria_s: float = 5.0     # la programación dinámica elige entre las 2 soluciones mirando los últimos 5 s
    ucmc_calib_min_cobertura: float = 30.0  # % de ventanas válidas en esa memoria para fiarse de la H
    ucmc_calib_min_muestras: int = 4      # ...y al menos este número de muestras válidas
    ucmc_calib_suavizado: int = 5         # mediana de las últimas 5 muestras (quita saltos aislados)
    ucmc_calib_max_salto: float = 20.0    # °/s: un cambio mayor se cuenta como "salto" (diagnóstico)
    ucmc_calib_altura_memoria_s: float = 10.0  # vehículos de los últimos 10 s fijan la escala de la altura
    ucmc_calib_min_vehiculos: int = 5     # vehículos mínimos para fiarse de esa escala


@dataclass
class CamaraUCMC:
    """Lo que se sabe de la cámara de ANTEMANO (por dron / por video)."""
    hfov_deg: float = 82.0           # campo de visión horizontal: DATO de la cámara (DJI típicos: 73-84°). No se estima
    pitch_respaldo_deg: float = 0.0  # 0 = cenital, 90 = horizonte al centro. Se usa hasta la 1ª medida (o siempre, en manual)
    altura_m: float = None           # None = automática (vehículos + H); un número = altura fija sobre el suelo
    nota: str = ""                   # de dónde salió (para no olvidar qué es estimado)

## PARTE 2 — Motor compartido (igual para los cuatro trackers)
Lo que va **antes** del tracker. Es idéntico en todos para que la comparación sea justa.

### 2.1 Escala
**Entra:** frame original. **Sale:** frame de trabajo (960 px) y su gris suavizado. También pasa las cajas de ida (detecciones → trabajo) y de vuelta (salida → original).

In [ ]:
# ======================================================================
# 2.1 · Escala
# Qué hace : Pasa el frame original a la resolución de trabajo, y las cajas de ida (detecciones) y vuelta (salida)
# ¿Se toca? : No (work_width en Config)
# ======================================================================
class Escala:
    """Entra: frame BGR original. Sale: frame de trabajo (BGR) y su gris suavizado (lo usan la H y las propuestas).
    El factor se fija con el primer frame; si cambia la resolución del stream, se crea un tracker nuevo."""

    def __init__(self, cfg, forma_frame):
        H0, W0 = forma_frame[:2]
        self.k = cfg.work_width / W0                                     # original -> trabajo
        self.ww, self.wh = cfg.work_width, int(round(H0 * self.k))

    def frame(self, frame_bgr):
        small = cv2.resize(frame_bgr, (self.ww, self.wh), interpolation=cv2.INTER_AREA)
        gray = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (3, 3), 0)
        return small, gray

    def a_trabajo(self, dets):
        d = np.asarray(dets, np.float32).reshape(-1, 6).copy()
        d[:, :4] *= self.k
        return d

    def a_original(self, caja):
        return np.asarray(caja, np.float32) / self.k

### 2.2 Estabilizador (movimiento de la cámara)
**Entra:** gris del frame actual. **Sale:** `(H, ok)`.

`H` lleva un punto del frame **anterior** al **actual** suponiendo que todo es suelo: un objeto quieto en el suelo queda en `H·(posición anterior)`. Se estima con LK + RANSAC sobre puntos repartidos en una rejilla (ninguna zona domina). `ok=False` si no hay puntos coherentes suficientes (borroso, agua, cambio brusco).

In [ ]:
# ======================================================================
# 2.2 · Estabilizador
# Qué hace : Estima cómo se movió la cámara entre el frame anterior y el actual (homografía H, LK + RANSAC)
# ¿Se toca? : No (parámetros en Config → [CAMARA])
# ======================================================================
class Estabilizador:
    """Entra: gris del frame actual. Sale: (H, ok).
    H lleva un punto del frame ANTERIOR al ACTUAL suponiendo que todo es suelo: un objeto quieto en el suelo queda en
    H·(posición anterior). ok=False si no hay suficientes puntos coherentes (imagen borrosa, agua, cambio brusco):
    en ese caso H = identidad y los historiales se reinician."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.prev = None

    def _puntos(self, gray):
        """Puntos repartidos en una rejilla, para que ninguna zona (techos, pastizal) domine la homografía."""
        h, w = gray.shape
        gx, gy = self.cfg.grid
        pts = []
        for i in range(gy):
            for j in range(gx):
                y0, y1, x0, x1 = i * h // gy, (i + 1) * h // gy, j * w // gx, (j + 1) * w // gx
                p = cv2.goodFeaturesToTrack(gray[y0:y1, x0:x1], 150, 0.01, 8)
                if p is not None:
                    pts.append(p.reshape(-1, 2) + np.array([x0, y0], np.float32))
        return np.vstack(pts).reshape(-1, 1, 2).astype(np.float32) if pts else None

    def paso(self, gray):
        if self.prev is None:                                            # primer frame: no hay movimiento que medir
            self.prev = gray
            return np.eye(3), True
        H, ok = np.eye(3), False
        p0 = self._puntos(self.prev)
        if p0 is not None and len(p0) >= 10:
            p1, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, gray, p0, None, winSize=(21, 21), maxLevel=3)
            good = st.ravel() == 1
            if good.sum() >= 10:
                Hc, inl = cv2.findHomography(p0[good], p1[good], cv2.RANSAC, self.cfg.ransac_px)
                if Hc is not None and inl.sum() >= self.cfg.min_inliers:
                    H, ok = Hc, True
        self.prev = gray
        return H, ok

### 2.3 Entrada de detecciones (+ propuestas)
**Entra:** detecciones del detector externo (píxeles originales). **Sale:** detecciones en píxeles de trabajo, sin cajas gigantes (`max_box_area_frac`), más las **propuestas**.

- `EntradaDetecciones.revisar` avisa con un mensaje claro si las cajas llegan como `[x, y, ancho, alto]`.
- **Propuestas:** diferencia con el frame de hace 0.15 s ya compensado con `H` → cajas **sin clase** (-1) donde algo se mueve y el detector no ve nada. Se descartan las que caen sobre una detección reciente (su estela) y las estructuras de una sola orientación (cables, bordillos).

In [ ]:
# ======================================================================
# 2.3 · EntradaDetecciones (+ Propuestas)
# Qué hace : Revisa y escala las detecciones del detector externo; opcionalmente añade "propuestas" sin clase
# ¿Se toca? : No (parámetros en Config → [ENTRADA], [PROPUESTAS])
# ======================================================================
class Propuestas:
    """Cambios respecto al frame t-gap, ya compensando la cámara con H: cajas SIN clase (clase -1) para cosas que se
    mueven y el detector no conoce. Se descartan las que caen sobre una detección reciente (su estela)."""

    def __init__(self, cfg, fps):
        self.cfg = cfg
        self.gap = max(2, int(round(cfg.proposal_gap_sec * fps)))
        self.grays = deque(maxlen=self.gap + 1)
        self.Hs = deque(maxlen=self.gap)
        self.oks = deque(maxlen=self.gap)
        self.memoria = deque(maxlen=max(1, int(round(cfg.proposal_memory_sec * fps))))   # cajas recientes, una entrada por frame

    def paso(self, gray, H, ok, dets):
        """dets: detecciones del frame (trabajo). Devuelve las propuestas (Nx6) ya filtradas."""
        self._recordar(dets, H)
        pr = self._cambios(gray, H, ok)
        pr = self._fuera_de_memoria(pr)
        if len(pr) and len(dets):
            pr = pr[iou_pad(pr[:, :4], dets[:, :4], 2.0).max(1) < 0.1]
        return pr

    def _recordar(self, dets, H):
        """Lleva las cajas recordadas al frame actual con H y añade las de este frame."""
        self.memoria = deque([warp_pts(H, b.reshape(-1, 2)).reshape(-1, 4) for b in self.memoria], maxlen=self.memoria.maxlen)
        if len(dets):
            self.memoria.append(dets[:, :4].astype(np.float32).copy())

    def _cambios(self, gray, H, ok):
        c = self.cfg
        if len(self.grays) > 0:
            self.Hs.append(H); self.oks.append(ok)
        self.grays.append(gray)
        if len(self.grays) < self.gap + 1 or not all(self.oks):
            return np.zeros((0, 6), np.float32)
        M = np.eye(3)
        for h in self.Hs:                                                # H_t @ ... @ H_{t-gap+1}
            M = h @ M
        prev, curr = self.grays[0], gray
        h_, w_ = curr.shape
        warped = cv2.warpPerspective(prev, M, (w_, h_))
        valid = cv2.erode(cv2.warpPerspective(np.full((h_, w_), 255, np.uint8), M, (w_, h_)), np.ones((11, 11), np.uint8))
        d = cv2.bitwise_and(cv2.absdiff(curr, warped), valid)
        if valid.any() and d[valid > 0].mean() > c.max_residual:
            return np.zeros((0, 6), np.float32)                          # la alineación falló: todo "cambió"
        m = (d > c.diff_thr).astype(np.uint8) * 255
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
        m = cv2.dilate(m, np.ones((c.dilate, c.dilate), np.uint8))
        n, _, s, _ = cv2.connectedComponentsWithStats(m)
        gx = cv2.Sobel(curr, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(curr, cv2.CV_32F, 0, 1, ksize=3)
        out = []
        for k in range(1, n):
            x, y, bw, bh, a = s[k]
            if not (c.min_area <= a <= c.max_area):
                continue
            if x < c.border or y < c.border or x + bw > w_ - c.border or y + bh > h_ - c.border:
                continue
            ar = bw / max(bh, 1)
            if ar > c.max_aspect or ar < 1 / c.max_aspect:
                continue
            sx, sy = gx[y:y + bh, x:x + bw], gy[y:y + bh, x:x + bw]
            jxx, jyy, jxy = float((sx * sx).sum()), float((sy * sy).sum()), float((sx * sy).sum())
            coh = np.sqrt((jxx - jyy) ** 2 + 4 * jxy ** 2) / (jxx + jyy + 1e-6)
            if coh > c.max_coherence:
                continue                                                 # línea/cable/borde, no un objeto
            out.append([x, y, x + bw, y + bh, c.proposal_conf, -1])
        return np.array(out, np.float32).reshape(-1, 6)

    def _fuera_de_memoria(self, pr):
        """Quita las propuestas cuya área cae en >= proposal_overlap sobre alguna detección recordada."""
        if not len(pr) or not self.memoria:
            return pr
        D = np.vstack(self.memoria)
        x1 = np.maximum(pr[:, None, 0], D[None, :, 0]); y1 = np.maximum(pr[:, None, 1], D[None, :, 1])
        x2 = np.minimum(pr[:, None, 2], D[None, :, 2]); y2 = np.minimum(pr[:, None, 3], D[None, :, 3])
        inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
        area = np.maximum((pr[:, 2] - pr[:, 0]) * (pr[:, 3] - pr[:, 1]), 1.0)
        return pr[(inter / area[:, None]).max(1) < self.cfg.proposal_overlap]


class EntradaDetecciones:
    """Entra: detecciones del detector EXTERNO, Nx6 [x1, y1, x2, y2, conf, clase] en píxeles del frame ORIGINAL
    (clase = id entero >= 0). Sale: detecciones en píxeles de TRABAJO, sin cajas gigantes, + propuestas (clase -1)."""

    def __init__(self, cfg, fps):
        self.cfg = cfg
        self.propuestas = Propuestas(cfg, fps) if cfg.use_proposals else None

    @staticmethod
    def revisar(dets):
        """Convierte a Nx6 float32 y avisa con un mensaje claro si el formato no es el esperado."""
        if dets is None:
            return np.zeros((0, 6), np.float32)
        d = np.asarray(dets, np.float32)
        if d.size == 0:
            return np.zeros((0, 6), np.float32)
        if d.ndim != 2 or d.shape[1] < 6:
            raise ValueError(f"Las detecciones deben ser Nx6 [x1, y1, x2, y2, conf, clase]; llegó {d.shape}")
        d = d[:, :6]
        if np.any(d[:, 2] < d[:, 0]) or np.any(d[:, 3] < d[:, 1]):
            raise ValueError("Cajas con x2 < x1 o y2 < y1: ¿el detector devuelve [x, y, ancho, alto]? Hay que pasarlas a esquinas.")
        return d

    def preparar(self, dets, escala, gray, H, ok):
        d = escala.a_trabajo(self.revisar(dets))
        if len(d):
            area = (d[:, 2] - d[:, 0]) * (d[:, 3] - d[:, 1])
            d = d[area <= self.cfg.max_box_area_frac * escala.ww * escala.wh]
        if self.propuestas is not None:
            pr = self.propuestas.paso(gray, H, ok, d)
            if len(pr):
                d = np.vstack([d, pr]) if len(d) else pr
        return d

### 2.4 Track
Objeto `track` común: caja, votos de clase, historial de centros y estado de movimiento.

El historial está siempre **en coordenadas del frame actual**: cada frame se re-proyecta con `H`. Así un objeto quieto tiene un historial quieto aunque el dron se mueva. La clase es la más votada (solo votan detecciones con `conf ≥ 0.40`, y hacen falta 3 votos).

In [ ]:
# ======================================================================
# 2.4 · Track
# Qué hace : Objeto 'track' común: caja, votos de clase, historial ESTABILIZADO y estado de movimiento
# ¿Se toca? : No
# ======================================================================
class Track:
    """El historial de centros (pts) está siempre en coordenadas del frame ACTUAL: cada frame se re-proyecta con H.
    Así un objeto quieto en el suelo tiene un historial quieto aunque el dron se mueva."""

    def __init__(self, tid, det, fidx, maxlen, cls_min_conf=0.40, cls_min_hits=3):
        self.id = tid
        self.box = det[:4].astype(np.float32).copy()
        self.conf = float(det[4])
        self.hits, self.miss = 1, 0
        self.cls_min_conf, self.cls_min_hits = cls_min_conf, cls_min_hits
        self.votes = defaultdict(float)
        self.counts = defaultdict(int)
        self._vote(det)
        self.pts = np.array([self._center()], np.float32)   # historial de centros (frame actual)
        self.gs = np.zeros((1, 2), np.float32)             # flujo que la cámara induce en el track, por frame (paralaje)
        self._g = np.zeros(2, np.float32)
        self.maxlen = maxlen
        self.first = self.last = fidx
        # estado de movimiento (lo llena DecisorMovimiento)
        self.disp, self.vec, self.moving = 0.0, np.zeros(2, np.float32), False
        self.straight, self.par, self.k_par = 1.0, False, None
        self.on_cnt = self.off_cnt = 0
        self.toggles = self.moving_frames = self.par_frames = 0
        self.max_disp = 0.0

    def _center(self):
        return [(self.box[0] + self.box[2]) / 2, (self.box[1] + self.box[3]) / 2]

    def _vote(self, det):
        if det[5] >= 0 and det[4] >= self.cls_min_conf:
            self.votes[int(det[5])] += float(det[4]); self.counts[int(det[5])] += 1

    @property
    def cls(self):
        """Clase más votada (suma de confianzas) si tiene al menos cls_min_hits votos; si no, -1 (desconocido)."""
        if not self.votes:
            return -1
        c = max(self.votes, key=self.votes.get)
        return c if self.counts[c] >= self.cls_min_hits else -1

    def warp(self, H, H_ok=True):
        """Lleva caja e historial al frame actual con H. Sin H fiable, el historial se reinicia (no es comparable)."""
        old_c = np.array(self._center(), np.float32)
        self.box = warp_pts(H, [self.box[:2], self.box[2:]]).reshape(-1).astype(np.float32)
        self._g = np.array(self._center(), np.float32) - old_c          # flujo del suelo en este punto
        self.pts = warp_pts(H, self.pts)
        A = (H[:2, :2] / H[2, 2]).astype(np.float32)
        self.gs = self.gs @ A.T
        if not H_ok:
            self.pts, self.gs = self.pts[-1:], self.gs[-1:]

    def update(self, det, fidx):
        self.box = det[:4].astype(np.float32).copy()
        self.conf = float(det[4])
        self.hits += 1; self.miss = 0; self.last = fidx
        self._vote(det)
        self.pts = np.vstack([self.pts, self._center()])[-self.maxlen:]
        self.gs = np.vstack([self.gs, self._g])[-self.maxlen:]

### 2.5 Libro de tracks (solo trackers externos)
**Entra:** la salida del tracker externo, `Nx7 [x1, y1, x2, y2, id, conf, clase]`. **Sale:** objetos `Track` confirmados.

Convierte "IDs + cajas" en `Track` con historial estabilizado, para que la decisión de movimiento (PARTE 4) sea **exactamente** la misma que en el tracker propio.

In [ ]:
# ======================================================================
# 2.5 · LibroTracks
# Qué hace : Para trackers EXTERNOS (UCMC, ByteTrack, BoT-SORT): convierte su salida (ids + cajas) en objetos Track
# ¿Se toca? : No
# ======================================================================
class LibroTracks:
    """Entra: salida del tracker externo, Nx7 [x1, y1, x2, y2, id, conf, clase] (trabajo). Sale: Track confirmados.
    Así la decisión de movimiento y la salida son EXACTAMENTE las mismas que en el tracker propio."""

    def __init__(self, cfg, fps):
        self.cfg = cfg
        self.tracks = {}
        self.max_miss = max(2, int(round(cfg.max_miss_sec * fps)))
        self.maxlen = int(round(cfg.window_sec * fps)) + 1

    def actualizar(self, out, H, H_ok, fidx):
        for t in self.tracks.values():
            t.warp(H, H_ok)
        vistos = set()
        for r in out:
            tid = int(r[4])
            det = np.array([r[0], r[1], r[2], r[3], r[5], r[6]], np.float32)
            if tid in self.tracks:
                self.tracks[tid].update(det, fidx)
            else:
                self.tracks[tid] = Track(tid, det, fidx, self.maxlen, self.cfg.cls_min_conf, self.cfg.cls_min_hits)
            vistos.add(tid)
        for tid in list(self.tracks):
            if tid not in vistos:
                self.tracks[tid].miss += 1
                if self.tracks[tid].miss > self.max_miss:
                    del self.tracks[tid]
        return [t for t in self.tracks.values() if t.hits >= self.cfg.min_hits and t.miss == 0]

## PARTE 3 — El tracker UCMC
### 3.1 Mapa (imagen ↔ suelo)
Para una cámara `(K, pitch, altura)`: `A = K·[r1 r2 t]` lleva `(x, y, 1)` del suelo (metros) a `(u, v, 1)` de la imagen. `uv2xy` también propaga la incertidumbre de la detección (píxeles → metros) con el jacobiano, como el original. `K` sale del **hfov** (dato).

In [ ]:
# ======================================================================
# 3.1 · Mapa (imagen <-> suelo)
# Qué hace : Proyecta píxeles al plano del suelo (metros) y al revés, para una cámara (K, pitch, altura)
# ¿Se toca? : No
# ======================================================================
def matriz_K(ww, wh, hfov_deg):
    """Intrínsecos de una cámara ideal (centro óptico en el centro de la imagen) con ese hfov, en píxeles de trabajo."""
    f = (ww / 2.0) / np.tan(np.deg2rad(hfov_deg) / 2.0)
    return np.array([[f, 0, ww / 2.0], [0, f, wh / 2.0], [0, 0, 1.0]]), f


class Mapa:
    """Como detector/mapper.py del original, vectorizado. A = K·[r1 r2 t] lleva (x, y, 1) del suelo (m) a (u, v, 1)."""

    def __init__(self, K, pitch_deg, altura_m):
        self.pitch_deg, self.altura_m = float(pitch_deg), float(altura_m)
        th = np.deg2rad(pitch_deg)
        # Mundo: X derecha, Y adelante, Z arriba. Cámara: x derecha, y abajo, z = eje óptico.
        R = np.array([[1.0, 0.0, 0.0],
                      [0.0, -np.cos(th), -np.sin(th)],
                      [0.0, np.sin(th), -np.cos(th)]])
        T = -R @ np.array([0.0, 0.0, float(altura_m)])                  # la cámara está a altura_m sobre el suelo
        self.A = K @ np.column_stack([R[:, 0], R[:, 1], T])
        self.InvA = np.linalg.inv(self.A)

    def uv2xy(self, uv, sigma_uv):
        """uv: Nx2 píxeles; sigma_uv: Nx2x2. Devuelve xy (Nx2, metros), sigma_xy (Nx2x2) y válido (N,)."""
        uv1 = np.column_stack([uv, np.ones(len(uv))])
        b = uv1 @ self.InvA.T
        valid = b[:, 2] > 1e-9                                           # por encima del horizonte: no corta el suelo
        gamma = 1.0 / np.where(valid, b[:, 2], 1.0)
        xy = b[:, :2] * gamma[:, None]
        # Jacobiano de la proyección (igual que el original): C = γ·InvA[:2,:2] − γ²·b[:2]·InvA[2,:2]
        C = gamma[:, None, None] * self.InvA[None, :2, :2] - (gamma ** 2)[:, None, None] * b[:, :2, None] * self.InvA[None, 2:3, :2]
        return xy, C @ sigma_uv @ np.transpose(C, (0, 2, 1)), valid

    def xy2uv(self, xy):
        p = np.column_stack([xy, np.ones(len(xy))]) @ self.A.T
        return p[:, :2] / p[:, 2:3]

### 3.2 `CalibradorCamara` — lo que pasa de manual a automático
Se llama **una vez por frame** con la `H` y las detecciones. Devuelve el mapa de ahora y el del frame anterior.

**Pitch, con la homografía del suelo.**
1. Cada 0.5 s se toma la `H` de la última ventana de 1 s y se descompone como `H = K (R + t·nᵀ/d) K⁻¹`. `n` es la normal del suelo vista desde la cámara: `pitch = atan2(n_y, n_z)`. `H` da la normal al inicio de la ventana y `H⁻¹`, ahora.
2. Se descartan las soluciones imposibles: suelo detrás de la cámara, dron casi quieto (`|t| < 0.03`), alabeo > 25°, pitch fuera de [-3°, 88°].
3. **Techo de horizonte:** con pitch θ el horizonte queda en `y = -cot θ`. Ningún vehículo puede apoyarse por encima, así que el 3er vehículo más alto limita θ.
4. Suelen quedar **2 soluciones** por muestra. Una **programación dinámica** sobre los últimos 5 s elige la secuencia más continua en el tiempo.
5. **Pitch de ahora** = mediana de las últimas 5 muestras de esa secuencia.
6. **Pocas muestras válidas** (< 30 % o < 4; hover, `H` no fiable): se conserva el último pitch, o `pitch_respaldo_deg` si todavía no hubo ninguno.

**Altura: la forma con la `H` y la escala con los vehículos.**

7. La misma descomposición da cuánto cambió la distancia al suelo: `d_b/d_a = 1 + (R·n)·t/d`. Encadenando, se conoce `log(altura)` salvo una constante.
8. Cada vehículo cerca del centro da una altura: `f · 4.5 m / lado_px · cos(pitch)`. La mediana de los últimos 10 s fija la constante (cada vehículo se corrige por lo que el dron subió o bajó desde que se vio).

**Costo de ser causal:** el pitch va **~1 s atrasado** (con el gimbal girando 3°/s, ~4.5° de error mientras gira). En los videos VIGIA el pitch cambia mucho más despacio.

**Estado publicado** (para diagnóstico): `pitch`, `altura`, `fuente_pitch` (`"respaldo"` / `"H"`), `fuente_altura` (`"defecto"` / `"vehiculos"` / `"fija"`), `n_saltos`.

In [ ]:
# ======================================================================
# 3.2 · CalibradorCamara
# Qué hace : Estima EN LÍNEA el pitch (con la H) y la altura (con la H + los vehículos) de cada frame
# ¿Se toca? : No (parámetros en ConfigUCMC → [CALIBRACION EN LINEA], [ALTURA])
# ======================================================================
class CalibradorCamara:
    """
    PITCH (inclinación) — con la homografía del suelo
      La H entre dos frames separados 1 s (ventana) se descompone como H = K (R + t·nᵀ/d) K⁻¹ (suelo plano).
      n es la normal del suelo vista desde la cámara: cenital -> (0, 0, 1); inclinada θ -> (0, sin θ, cos θ),
      así que  pitch = atan2(n_y, n_z)  y  roll = asin(n_x).
        a) Cada 0.5 s se descompone la H de la última ventana [a, b]: H da la normal en a; H⁻¹, en b (ahora).
        b) Se descartan las soluciones imposibles: suelo detrás de la cámara (n_z <= 0), dron casi quieto (|t| pequeño,
           la normal no se puede medir), alabeo > 25°, pitch fuera de [-3°, 88°].
        c) "Techo" de horizonte: con pitch θ el horizonte queda en y = -cot θ (y = (v - cy)/f). Ningún vehículo puede
           apoyarse por encima, así que la base del 3er vehículo más alto limita θ.
        d) Quedan normalmente 2 soluciones por muestra. Programación dinámica sobre las muestras de los últimos
           `ucmc_calib_memoria_s`: se elige la secuencia más CONTINUA en el tiempo (mínimo salto de pitch).
        e) El pitch de ahora = mediana de las últimas `ucmc_calib_suavizado` muestras de esa secuencia. Costo de ser
           causal: va ~1 s atrasado (si el gimbal gira 3°/s, ~3° de error mientras gira).
        f) Si en la memoria hay pocas muestras válidas (dron en hover, H no fiable), se conserva el último pitch
           (o pitch_respaldo_deg si todavía no hubo ninguno).
    ALTURA — forma con la H, escala con los vehículos
        g) La misma descomposición da cuánto cambió la distancia al suelo: d_b/d_a = 1 + (R·n)·(t/d).
           Encadenando, log(altura) se conoce salvo una constante: L(t).
        h) Cada vehículo detectado cerca del centro da una altura: f · largo_vehiculo / lado_px · cos(pitch).
           Se guarda "relativo a L" y la mediana de los últimos `ucmc_calib_altura_memoria_s` fija la constante:
           altura(ahora) = exp(L(ahora) + mediana(log altura_vehículo - L(cuando se vio))).
           Así un vehículo visto hace 5 s sigue sirviendo aunque el dron haya bajado desde entonces.
    """

    def __init__(self, cfg, fps, cam, ww, wh):
        self.cfg, self.cam = cfg, cam
        self.K, self.f = matriz_K(ww, wh, cam.hfov_deg)
        self.ww, self.wh = ww, wh
        self.gap = max(2, int(round(cfg.ucmc_calib_ventana_s * fps)))   # frames de la ventana
        self.paso = max(1, int(round(fps / 2)))                          # una ventana cada medio segundo
        self.memoria = int(round(cfg.ucmc_calib_memoria_s * fps))
        self.memoria_alt = int(round(cfg.ucmc_calib_altura_memoria_s * fps))
        self.fps = fps
        self.Hs, self.oks = deque(maxlen=self.gap), deque(maxlen=self.gap)
        self.k = -1                                                      # índice del frame actual
        self.muestras = []                                               # [frame, candidatos, es_inversa, (a, b)]
        self.techos = {}                                                 # frame -> techo de pitch (últimos gap frames)
        self.techo = None                                                # último techo conocido
        self.L = {}                                                      # frame -> log(altura) relativo (últimos gap frames)
        self.L_ahora = 0.0
        self.vehiculos = deque()                                         # (frame, log altura_vehículo - L)
        # estado publicado
        self.pitch = float(cam.pitch_respaldo_deg)
        self.altura = float(cam.altura_m) if cam.altura_m is not None else float(cfg.ucmc_altura_defecto_m)
        self.fuente_pitch = "respaldo"
        self.fuente_altura = "fija" if cam.altura_m is not None else "defecto"
        self.n_saltos = 0
        self.mapa = Mapa(self.K, self.pitch, self.altura)

    # ---- entrada principal ---------------------------------------------
    def actualizar(self, H, H_ok, dets):
        """Llamar UNA vez por frame. dets: detecciones de trabajo (Nx6). Devuelve (mapa de ahora, mapa del frame anterior)."""
        self.k += 1
        mapa_prev = self.mapa
        if not self.cfg.ucmc_calibrar:                                   # modo MANUAL
            return self.mapa, mapa_prev
        c = self.cfg
        vehiculos = dets[(dets[:, 5] >= 0) & (dets[:, 4] >= c.conf_high)] if len(dets) else dets
        self._techo_de(vehiculos)                                        # c)
        self.Hs.append(H); self.oks.append(H_ok)
        a = self.k - self.gap
        L_nuevo = self.L_ahora
        if a >= 0 and a % self.paso == 0:                                # a) toca una ventana
            v_alt = self._muestrear(a)
            if v_alt:                                                    # g)
                L_nuevo = self.L.get(a, self.L_ahora) + float(np.mean(v_alt))
        self.L_ahora = L_nuevo
        self.L[self.k] = L_nuevo
        for fr in [f for f in self.L if f < self.k - self.gap - 1]:
            del self.L[fr]
        if self.cam.altura_m is None:
            self._altura_con(vehiculos)                                  # h)
        if abs(self.pitch - self.mapa.pitch_deg) > 0.05 or abs(self.altura - self.mapa.altura_m) > 0.002 * self.mapa.altura_m:
            self.mapa = Mapa(self.K, self.pitch, self.altura)            # la cámara cambió: nuevo mapeo
        return self.mapa, mapa_prev

    # ---- c) techo de horizonte -------------------------------------------
    def _techo_de(self, vehiculos):
        if len(vehiculos) >= 3:
            y = np.sort((vehiculos[:, 3] - self.wh / 2.0) / self.f)[2]  # base del 3er vehículo más alto
            self.techo = 90.0 if y >= 0 else float(np.degrees(np.arctan(1.0 / -y)))
        self.techos[self.k] = self.techo
        for fr in [f for f in self.techos if f < self.k - self.gap - 1]:
            del self.techos[fr]

    # ---- a) b) d) e) f) pitch ----------------------------------------------
    def _candidatos(self, H):
        """[(pitch, roll, |t|/d, d2/d1)] físicamente posibles de H = K (R + t·nᵀ/d) K⁻¹."""
        c = self.cfg
        n_sol, Rs, ts, ns = cv2.decomposeHomographyMat(H / H[2, 2], self.K)
        out = []
        for k in range(n_sol):
            n, t = ns[k].ravel(), ts[k].ravel()
            if not np.all(np.isfinite(n)) or n[2] <= 0:                  # suelo detrás de la cámara
                continue
            if np.linalg.norm(t) < c.ucmc_calib_min_t:                   # casi sin desplazamiento: no se puede medir
                continue
            p = float(np.degrees(np.arctan2(n[1], n[2])))
            r = float(np.degrees(np.arcsin(np.clip(n[0], -1, 1))))
            ratio = 1.0 + float((Rs[k] @ n) @ t)
            if -3.0 <= p <= 88.0 and abs(r) <= c.ucmc_calib_max_roll and ratio > 0:
                out.append((max(p, 0.0), r, float(np.linalg.norm(t)), ratio))
        return out

    def _muestrear(self, a):
        """Descompone la H de la ventana [a, ahora], resuelve el pitch y devuelve las medidas de log(d_ahora/d_a)."""
        b = self.k
        if all(self.oks):
            Hw = np.eye(3)
            for h in self.Hs:                                            # H de a hasta b (los últimos gap pasos)
                Hw = h @ Hw
            for fr, Hv, inv in ((a, Hw, False), (b, np.linalg.inv(Hw), True)):
                cands = self._candidatos(Hv)
                techo = self.techos.get(fr)
                if techo is not None:
                    cands = [x for x in cands if x[0] <= techo + 3.0]
                self.muestras.append([fr, cands, inv, (a, b)])
        else:                                                            # H no fiable en la ventana: muestra vacía
            self.muestras += [[a, [], False, (a, b)], [b, [], True, (a, b)]]
        self.muestras = sorted([m for m in self.muestras if m[0] >= self.k - self.memoria], key=lambda m: m[0])
        con = [m for m in self.muestras if m[1]]
        cobertura = 100.0 * len(con) / max(len(self.muestras), 1)
        if len(con) < self.cfg.ucmc_calib_min_muestras or cobertura < self.cfg.ucmc_calib_min_cobertura:
            return []                                                    # f) pocas muestras: se conserva el pitch
        sel = self._mas_continua(con)                                    # d)
        pitches = [m[1][s][0] for m, s in zip(con, sel)]
        nuevo = float(np.median(pitches[-self.cfg.ucmc_calib_suavizado:]))   # e)
        if abs(nuevo - self.pitch) * self.fps / self.paso > self.cfg.ucmc_calib_max_salto and self.fuente_pitch == "H":
            self.n_saltos += 1
        if self.fuente_pitch == "respaldo":                              # 1ª medida: las alturas de los vehículos
            self.vehiculos.clear()                                       # se calcularon con el pitch de respaldo
        self.pitch, self.fuente_pitch = nuevo, "H"
        v = []                                                           # g) cambio de altura en esta ventana
        for m, s in zip(con, sel):
            if m[3] == (a, b):
                r = m[1][s][3]
                x = -np.log(r) if m[2] else np.log(r)                    # H⁻¹ da d_a/d_b
                if abs(x) < 0.5:
                    v.append(x)
        return v

    @staticmethod
    def _mas_continua(con):
        """Programación dinámica: un candidato por muestra minimizando la suma de |Δpitch| entre muestras seguidas."""
        costo = np.zeros(len(con[0][1]))
        previo = []
        for k in range(1, len(con)):
            a = np.array([x[0] for x in con[k - 1][1]]); b = np.array([x[0] for x in con[k][1]])
            M = costo[:, None] + np.abs(a[:, None] - b[None, :])
            previo.append(M.argmin(0)); costo = M.min(0)
        sel = [int(costo.argmin())]
        for k in range(len(con) - 1, 0, -1):
            sel.append(int(previo[k - 1][sel[-1]]))
        return sel[::-1]

    # ---- h) altura -----------------------------------------------------------
    def _altura_con(self, vehiculos):
        c = self.cfg
        if len(vehiculos):
            cx, cy = (vehiculos[:, 0] + vehiculos[:, 2]) / 2, (vehiculos[:, 1] + vehiculos[:, 3]) / 2
            cerca = (np.abs(cx - self.ww / 2) < 0.3 * self.ww) & (np.abs(cy - self.wh / 2) < 0.3 * self.wh)
            lado = np.maximum(vehiculos[cerca, 2] - vehiculos[cerca, 0], vehiculos[cerca, 3] - vehiculos[cerca, 1])
            alt = self.f * c.ucmc_largo_vehiculo_m / np.maximum(lado, 1.0) * np.cos(np.deg2rad(self.pitch))
            for x in np.log(alt):
                self.vehiculos.append((self.k, float(x) - self.L_ahora))
        while self.vehiculos and self.vehiculos[0][0] < self.k - self.memoria_alt:
            self.vehiculos.popleft()
        if len(self.vehiculos) >= c.ucmc_calib_min_vehiculos:
            self.altura = float(np.exp(self.L_ahora + np.median([v for _, v in self.vehiculos])))
            self.fuente_altura = "vehiculos"
        elif self.fuente_altura == "vehiculos":                          # sin vehículos recientes: la forma la da la H
            self.altura = float(self.altura * np.exp(self.L_ahora - self.L.get(self.k - 1, self.L_ahora)))

### 3.3 Núcleo UCMC (MIT, adaptado del repo original)
Kalman de velocidad constante **en el suelo** (estado `x, vx, y, vy` en metros) y asociación con **MMD** (Mahalanobis² + log|S|) en 3 pasos:
1. fuertes ↔ confirmados; 2. débiles ↔ el resto; 3. fuertes sobrantes ↔ tentativos.

Nace con una detección fuerte, se confirma tras 3 detecciones seguidas y muere tras 1 s costeando.

**Diferencias con el original:** umbrales de confianza y ventana de pérdida = los comunes (como ByteTrack/BoT-SORT); sin "head padding" ni interpolación (son post-procesos offline); la **CMC** usa nuestra `H` y, si la cámara cambió, el punto sale al frame anterior con el mapa anterior y vuelve al suelo con el nuevo.

In [ ]:
# ======================================================================
# 3.3 · NucleoUCMC
# Qué hace : UCMCTrack portado a numpy: Kalman en el suelo, CMC con nuestra H, asociación MMD en 3 pasos
# ¿Se toca? : No (parámetros en ConfigUCMC → [UCMC])
# ----------------------------------------------------------------------
# Adaptado de https://github.com/corfyi/UCMCTrack (commit 7851a49), archivos
# tracker/ucmc.py, tracker/kalman.py y detector/mapper.py.
# MIT License — Copyright (c) 2023 Kefu Yi
# Permission is hereby granted, free of charge, to any person obtaining a copy of this software and
# associated documentation files (the "Software"), to deal in the Software without restriction, including
# without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the
# following conditions: The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT
# LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO
# EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER
# IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE
# USE OR OTHER DEALINGS IN THE SOFTWARE.
# ======================================================================
def sigma_uv(h, cfg):
    """getUVError del original: 5 % de la altura de la caja, con topes. Los topes del original (2-13 px en u,
    2-10 px en v) son para 1920 px de ancho: se escalan a work_width."""
    k = cfg.work_width / 1920.0
    u = np.clip(cfg.ucmc_uv_ratio * h, 2 * k, 13 * k)
    v = np.clip(cfg.ucmc_uv_ratio * h, 2 * k, 10 * k)
    S = np.zeros((len(h), 2, 2)); S[:, 0, 0] = u ** 2; S[:, 1, 1] = v ** 2
    return S


def asignar_con_limite(cost, limite):
    """Equivale a lap.lapjv(cost, extend_cost=True, cost_limit=limite): asignación óptima donde dejar algo sin pareja
    cuesta limite/2 por lado. Devuelve (pares, filas sin pareja, columnas sin pareja)."""
    n, m = cost.shape
    if n == 0 or m == 0:
        return [], list(range(n)), list(range(m))
    BIG = 1e12
    M = np.full((n + m, m + n), BIG)
    M[:n, :m] = np.where(np.isfinite(cost), cost, BIG)
    M[:n, m:] = np.where(np.eye(n, dtype=bool), limite / 2.0, BIG)
    M[n:, :m] = np.where(np.eye(m, dtype=bool), limite / 2.0, BIG)
    M[n:, m:] = 0.0
    r, c = linear_sum_assignment(M)
    pares = [(i, j) for i, j in zip(r, c) if i < n and j < m and M[i, j] < BIG]
    pi = {i for i, _ in pares}; pj = {j for _, j in pares}
    return pares, [i for i in range(n) if i not in pi], [j for j in range(m) if j not in pj]


def mmd(trks, Y, Rs):
    """Mapped Mahalanobis Distance: d² de Mahalanobis + log|S|, detección x track."""
    if not trks or len(Y) == 0:
        return np.zeros((len(Y), len(trks)))
    zp = np.array([[t.x[0], t.x[2]] for t in trks])
    HPH = np.array([t.P[np.ix_([0, 2], [0, 2])] for t in trks])
    S = HPH[None, :, :, :] + Rs[:, None, :, :]
    d = Y[:, None, :] - zp[None, :, :]
    a, b, c, e = S[..., 0, 0], S[..., 0, 1], S[..., 1, 0], S[..., 1, 1]
    det = np.maximum(a * e - b * c, 1e-12)
    m = (e * d[..., 0] ** 2 - (b + c) * d[..., 0] * d[..., 1] + a * d[..., 1] ** 2) / det
    return m + np.log(det)


TENTATIVO, CONFIRMADO, COSTEANDO = 0, 1, 2       # TrackStatus del original: Tentative, Confirmed, Coasted


class TrackSuelo:
    """KalmanTracker del original: velocidad constante en el SUELO, estado (x, vx, y, vy) en metros."""

    def __init__(self, tid, y, wx, wy, vmax, dt):
        self.F = np.array([[1, dt, 0, 0], [0, 1, 0, 0], [0, 0, 1, dt], [0, 0, 0, 1]], np.float64)
        self.Hm = np.array([[1, 0, 0, 0], [0, 0, 1, 0]], np.float64)
        G = np.zeros((4, 2)); G[0, 0] = 0.5 * dt * dt; G[1, 0] = dt; G[2, 1] = 0.5 * dt * dt; G[3, 1] = dt
        self.Q = G @ np.diag([wx, wy]) @ G.T                             # ruido de proceso (aceleración)
        self.x = np.array([y[0], 0.0, y[1], 0.0])
        self.P = np.diag([1.0, vmax ** 2 / 3.0, 1.0, vmax ** 2 / 3.0])
        self.id, self.status = tid, TENTATIVO
        self.death_count = self.birth_count = 0
        self.detidx = -1

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

    def update(self, y, R):
        S = self.Hm @ self.P @ self.Hm.T + R
        K = self.P @ self.Hm.T @ np.linalg.inv(S)
        self.x = self.x + K @ (y - self.Hm @ self.x)
        I_KH = np.eye(4) - K @ self.Hm
        self.P = I_KH @ self.P @ I_KH.T + K @ R @ K.T                    # forma de Joseph, como filterpy


class NucleoUCMC:
    """Fiel al original salvo:
      - Umbrales de confianza y ventana de pérdida = los comunes de Config (igual que ByteTrack/BoT-SORT).
      - Si en un frame no hay detecciones débiles, el original deja los tracks confirmados sin pareja "confirmados"
        para siempre (solo borra los "costeando"). Aquí pasan a costear.
      - Sin "head padding" ni interpolación: son post-procesos OFFLINE del original (reescriben frames pasados).
      - CMC (su opción --cmc) alimentada con NUESTRA H; si la cámara cambió de pitch/altura, el punto sale al frame
        anterior con el mapa anterior y vuelve al suelo con el mapa nuevo."""

    def __init__(self, cfg, fps, punto):
        self.cfg, self.punto = cfg, punto
        self.dt = 1.0 / fps
        self.max_age = max(2, int(round(cfg.max_miss_sec * fps)))     # frames costeando antes de borrar
        self.trackers, self.nid = [], 1
        self.n_fuera = 0                                               # detecciones por encima del horizonte (diagnóstico)
        self.ids_2det = set()                                          # IDs con >= 2 detecciones, aunque sigan tentativos
        self.mapa = None

    def _uv(self, b):
        u = (b[:, 0] + b[:, 2]) / 2
        v = b[:, 3] if self.punto == "base" else (b[:, 1] + b[:, 3]) / 2
        return np.column_stack([u, v])

    def _cmc(self, H, mapa_prev):
        """Lleva el punto de apoyo de cada track al frame actual con H (la velocidad no se toca, como en el original)."""
        if not self.trackers:
            return
        xy = np.array([[t.x[0], t.x[2]] for t in self.trackers])
        uv = warp_pts(H, mapa_prev.xy2uv(xy)).astype(np.float64)
        nxy, _, ok = self.mapa.uv2xy(uv, np.tile(np.eye(2), (len(uv), 1, 1)))
        for t, p, o in zip(self.trackers, nxy, ok):
            if o:
                t.x[0], t.x[2] = p

    def _asociar(self, idx_det, idx_trk, Y, Rs):
        C = mmd([self.trackers[k] for k in idx_trk], Y[idx_det], Rs[idx_det])
        pares, ud, ut = asignar_con_limite(C, self.cfg.ucmc_a)
        return [(idx_det[i], idx_trk[j]) for i, j in pares], [idx_det[i] for i in ud], [idx_trk[j] for j in ut]

    def _corregir(self, t, i, Y, Rs, status):
        t.update(Y[i], Rs[i])
        t.death_count = 0
        t.detidx = i
        if status is not None:
            t.status = status

    def actualizar(self, dets, H, H_ok, mapa, mapa_prev):
        """dets: Nx6 en píxeles de trabajo. Devuelve Nx7 [x1,y1,x2,y2,id,conf,clase]: tracks CONFIRMADOS con detección
        en este frame, con la caja de la detección (como el original)."""
        c = self.cfg
        self.mapa = mapa
        # a) detecciones al suelo (las que están por encima del horizonte no se pueden usar)
        dets = dets[dets[:, 4] >= c.conf_low] if len(dets) else np.zeros((0, 6), np.float32)
        if len(dets):
            Y, Rs, ok = mapa.uv2xy(self._uv(dets[:, :4]).astype(np.float64), sigma_uv(dets[:, 3] - dets[:, 1], c))
        else:
            Y, Rs, ok = np.zeros((0, 2)), np.zeros((0, 2, 2)), np.zeros(0, bool)
        self.n_fuera += int((~ok).sum())
        dets, Y, Rs = dets[ok], Y[ok], Rs[ok]
        fuertes = [i for i in range(len(dets)) if dets[i, 4] >= c.conf_high]
        debiles = [i for i in range(len(dets)) if dets[i, 4] < c.conf_high]
        # b) predicción + CMC
        for t in self.trackers:
            t.predict()
            t.detidx = -1
        if H_ok:
            self._cmc(H, mapa_prev)
        elif mapa_prev is not mapa:                                      # sin H fiable, pero cambió la cámara
            self._cmc(np.eye(3), mapa_prev)
        # c) asociaciones
        conf = [k for k, t in enumerate(self.trackers) if t.status in (CONFIRMADO, COSTEANDO)]
        pares, det_rest, trk_rest = self._asociar(fuertes, conf, Y, Rs)  # 1) fuertes vs confirmados
        for i, k in pares:
            self._corregir(self.trackers[k], i, Y, Rs, CONFIRMADO)
        pares2, _, trk_sin = self._asociar(debiles, trk_rest, Y, Rs)    # 2) débiles vs el resto
        for i, k in pares2:
            self._corregir(self.trackers[k], i, Y, Rs, CONFIRMADO)
        for k in trk_sin:
            self.trackers[k].status = COSTEANDO
        tent = [k for k, t in enumerate(self.trackers) if t.status == TENTATIVO]
        pares3, det_nuevas, _ = self._asociar(det_rest, tent, Y, Rs)    # 3) fuertes sobrantes vs tentativos
        for i, k in pares3:
            t = self.trackers[k]
            self._corregir(t, i, Y, Rs, None)
            self.ids_2det.add(t.id)
            t.birth_count += 1
            if t.birth_count >= 2:                                       # 3 detecciones seguidas -> confirmado
                t.birth_count, t.status = 0, CONFIRMADO
        # d) nacimientos (solo fuertes) y muertes
        for i in det_nuevas:
            t = TrackSuelo(self.nid, Y[i], c.ucmc_wx, c.ucmc_wy, c.ucmc_vmax, self.dt)
            t.detidx = i; self.nid += 1
            self.trackers.append(t)
        vivos = []
        for t in self.trackers:
            t.death_count += 1
            if (t.status == COSTEANDO and t.death_count >= self.max_age) or (t.status == TENTATIVO and t.death_count >= 2):
                continue
            vivos.append(t)
        self.trackers = vivos
        out = [[*dets[t.detidx, :4], t.id, dets[t.detidx, 4], dets[t.detidx, 5]]
               for t in self.trackers if t.status == CONFIRMADO and t.detidx >= 0]
        return np.array(out, np.float32).reshape(-1, 7)

### 3.4 `TrackerUCMC` (la clase principal)
`actualizar(frame, detecciones)` hace los 8 pasos; `seguir(dets, H, ok)` hace los pasos 4–6 (calibración → núcleo → libro). El calibrador se crea con el **primer frame**, porque necesita el tamaño del frame de trabajo.

In [ ]:
# ======================================================================
# 3.4 · TrackerUCMC
# Qué hace : El pipeline completo de un frame (pasos 1-8 del inicio del notebook), en una sola clase
# ¿Se toca? : No (parámetros en ConfigUCMC; la cámara en CamaraUCMC)
# ======================================================================
class TrackerUCMC:
    nombre = "ucmc"

    def __init__(self, fps, camara=None, cfg=None, nombres_clases=None):
        self.cfg = cfg or ConfigUCMC()
        self.camara = camara or CamaraUCMC()
        self.fps = float(fps)
        self.nombres_clases = dict(nombres_clases or {})
        c = self.cfg
        punto = c.ucmc_punto if c.ucmc_punto != "auto" else ("centro" if self.camara.pitch_respaldo_deg < 30 else "base")
        self.escala = None                                   # paso 1 (con el primer frame)
        self.calibrador = None                               # paso 4 (necesita el tamaño del frame de trabajo)
        self.estabilizador = Estabilizador(c)                # paso 2
        self.entrada = EntradaDetecciones(c, fps)            # paso 3
        self.nucleo = NucleoUCMC(c, fps, punto)              # paso 5
        self.libro = LibroTracks(c, fps)                     # paso 6
        self.decisor = DecisorMovimiento(c, fps, ids_sin_paralaje(self.nombres_clases, c))   # paso 7
        self.frame = 0
        self.cron = Cronometro()
        self.salida_nucleo = np.zeros((0, 7), np.float32)    # última salida de UCMC (diagnóstico / benchmarks)

    @property
    def tiempos_ms(self):
        return self.cron.ms

    @property
    def ids_emitidos(self):
        return self.nucleo.nid - 1

    def actualizar(self, frame_bgr, detecciones):
        """frame_bgr: frame ORIGINAL. detecciones: Nx6 [x1,y1,x2,y2,conf,clase] en píxeles del frame original."""
        self.cron.inicio()
        # 1. Escala
        if self.escala is None:
            self.escala = Escala(self.cfg, frame_bgr.shape)
            self.calibrador = CalibradorCamara(self.cfg, self.fps, self.camara, self.escala.ww, self.escala.wh)
        _, gray = self.escala.frame(frame_bgr)
        self.cron.marca("escala")
        # 2. Estabilización
        H, ok = self.estabilizador.paso(gray)
        self.cron.marca("estabilizacion")
        # 3. Entrada
        dets = self.entrada.preparar(detecciones, self.escala, gray, H, ok)
        self.cron.marca("entrada")
        # 4-6. Calibración + UCMC + libro
        confirmados = self.seguir(dets, H, ok)
        self.cron.marca("tracking")
        # 7. ¿Se mueve?
        self.decisor.actualizar(confirmados, ok)
        self.cron.marca("movimiento")
        # 8. Salida
        self.frame += 1
        return a_salida(confirmados, self.escala, self.nombres_clases)

    def seguir(self, dets, H, H_ok):
        """Pasos 4-6 sobre detecciones en píxeles de trabajo. Devuelve los Track confirmados de este frame."""
        mapa, mapa_prev = self.calibrador.actualizar(H, H_ok, dets)      # 4
        self.salida_nucleo = self.nucleo.actualizar(dets, H, H_ok, mapa, mapa_prev)   # 5
        return self.libro.actualizar(self.salida_nucleo, H, H_ok, self.frame)        # 6

## PARTE 4 — Decisión de movimiento y salida (igual para los cuatro)
### 4.1 ¿Se mueve o no?
Mide el **desplazamiento neto** del track en 1 s, en coordenadas estabilizadas, con tres defensas:
1. **Rectitud** (`min_straightness`): un objeto real avanza; el ruido de la caja deambula.
2. **Umbral relativo al tamaño** (`rel_move`): frena cajas grandes inestables.
3. **Paralaje:** si el desplazamiento es proporcional al flujo acumulado del suelo, es algo alto y quieto (poste, árbol). Los vehículos no pasan por este filtro.

Histéresis: 3 evaluaciones seguidas para encender, 10 para apagar.

In [ ]:
# ======================================================================
# 4.1 · DecisorMovimiento
# Qué hace : Decide si cada track se mueve DE VERDAD (no por el movimiento del dron)
# ¿Se toca? : No (parámetros en Config → [MOVIMIENTO], [PARALAJE])
# ======================================================================
def ids_sin_paralaje(nombres_clases, cfg):
    """Ids de las clases que no pasan por el filtro de paralaje (vehículos: no sobresalen del suelo como un poste)."""
    return {i for i, n in (nombres_clases or {}).items() if str(n).lower() in cfg.parallax_skip}


class DecisorMovimiento:
    """Mide el desplazamiento NETO del track en la ventana (window_sec) en coordenadas estabilizadas, con tres defensas:
      1) avanzar en línea (min_straightness): el ruido de la caja deambula, un objeto real avanza;
      2) umbral que crece con el tamaño del objeto (rel_move);
      3) paralaje: si el desplazamiento es proporcional al flujo acumulado del suelo, es algo alto y quieto (poste).
    Histéresis: on_frames evaluaciones seguidas para encender, off_frames para apagar."""

    def __init__(self, cfg, fps, skip_ids=()):
        self.cfg = cfg
        self.W = int(round(cfg.window_sec * fps))
        self.thr = cfg.min_move_px
        self.skip_ids = set(skip_ids)

    @staticmethod
    def _rectitud(p):
        """Desplazamiento neto / recorrido, sobre 6 tramos (medianas) para no contar el jitter de la caja."""
        if len(p) < 6:
            return 1.0
        m = np.array([np.median(ch, 0) for ch in np.array_split(p, 6)])
        recorrido = float(np.linalg.norm(np.diff(m, axis=0), axis=1).sum())
        return float(np.linalg.norm(m[-1] - m[0]) / (recorrido + 1e-6))

    def _es_paralaje(self, t, n):
        """Si el objeto está quieto en el mundo, su desplazamiento estabilizado P(t) es proporcional al flujo
        acumulado del suelo en ese punto C(t):  P(t) ≈ k·C(t).  k > 0: sobresale del suelo; k ≈ -1: fijo en la imagen.
        Se exige dirección paralela, k en [k_min, k_max] y buen ajuste en el tiempo (R²)."""
        c = self.cfg
        p, g = t.pts[-n:], t.gs[-n:]
        if len(p) < 6 or t.disp < 1e-6:
            t.k_par = None
            return False
        P = p - p[0]
        C = np.cumsum(g, 0); C = C - C[0]
        Gn = float(np.linalg.norm(C[-1]))
        if Gn < c.parallax_min_ground_px:
            t.k_par = None
            return False
        cos = float(t.vec @ C[-1]) / (t.disp * Gn)
        k = float((P * C).sum() / ((C * C).sum() + 1e-9))
        fit = 1.0 - float(((P - k * C) ** 2).sum()) / (float((P ** 2).sum()) + 1e-9)
        t.k_par = k
        return (abs(cos) >= np.cos(np.deg2rad(c.parallax_angle_deg))
                and c.parallax_k_min <= k <= c.parallax_k_max and fit >= c.parallax_min_fit)

    def actualizar(self, tracks, H_ok):
        c = self.cfg
        if not H_ok:                                                     # sin H fiable no se decide (se mantiene el estado)
            return
        disps = []
        for t in tracks:                                                 # 1) desplazamiento neto en la ventana
            n = min(len(t.pts), self.W + 1)
            if n < max(5, int(0.5 * self.W)):
                t.disp = 0.0; t.vec = np.zeros(2, np.float32); t.par = False; continue
            p = t.pts[-n:]; k = max(1, n // 5)
            t.vec = (np.median(p[-k:], 0) - np.median(p[:k], 0)).astype(np.float32)
            t.disp = float(np.linalg.norm(t.vec))
            disps.append(t.disp)
        self.thr = c.min_move_px
        if c.adaptive and len(disps) >= 5:
            med = np.median(disps); mad = np.median(np.abs(np.array(disps) - med)) * 1.4826
            self.thr = float(min(max(c.min_move_px, med + c.mad_k * mad), c.max_adaptive_px))
        for t in tracks:                                                 # 2) ¿cuenta como movimiento?
            n = min(len(t.pts), self.W + 1)
            if len(t.pts) < max(5, int(0.5 * self.W)):
                continue
            t.straight = self._rectitud(t.pts[-n:])
            size = float(np.sqrt(max((t.box[2] - t.box[0]) * (t.box[3] - t.box[1]), 1.0)))
            thr_t = max(self.thr, c.rel_move * size)
            bw, bh = t.box[2] - t.box[0], t.box[3] - t.box[1]
            estructura = t.cls < 0 and max(bw, bh) / max(min(bw, bh), 1.0) > c.max_unknown_aspect
            raw = t.disp > thr_t and t.straight >= c.min_straightness and not estructura
            t.par = False
            if raw and c.parallax_filter and t.cls not in self.skip_ids and self._es_paralaje(t, n):
                raw, t.par = False, True
                t.par_frames += 1
            if raw:                                                      # 3) histéresis
                t.on_cnt += 1; t.off_cnt = 0
            else:
                t.off_cnt += 1; t.on_cnt = 0
            if not t.moving and t.on_cnt >= c.on_frames:
                t.moving = True; t.toggles += 1
            elif t.moving and t.off_cnt >= c.off_frames:
                t.moving = False; t.toggles += 1
            t.moving_frames += int(t.moving)
            t.max_disp = max(t.max_disp, t.disp)

### 4.2 Salida
`TrackSalida` en píxeles **originales**, `como_array` para benchmarks MOT y `Cronometro` (ms por etapa: `tracker.tiempos_ms`).

In [ ]:
# ======================================================================
# 4.2 · Salida
# Qué hace : Lo que devuelve actualizar(): un TrackSalida por track confirmado, en píxeles del frame ORIGINAL
# ¿Se toca? : No
# ======================================================================
@dataclass
class TrackSalida:
    id: int
    caja: tuple                 # (x1, y1, x2, y2) en píxeles del frame original
    conf: float                 # confianza de la última detección asociada
    clase: int                  # id de clase votado (-1 = desconocido / propuesta)
    nombre_clase: str
    en_movimiento: bool         # decisión de DecisorMovimiento (con histéresis)
    desplazamiento_px: tuple    # desplazamiento neto en la ventana (≈ window_sec), estabilizado, px originales
    paralaje: bool = False      # True = parecía moverse pero lo explica el paralaje (algo alto y quieto)


def a_salida(tracks, escala, nombres_clases):
    nombres = nombres_clases or {}
    out = []
    for t in tracks:
        cls = t.cls
        out.append(TrackSalida(id=int(t.id), caja=tuple(float(x) for x in escala.a_original(t.box)), conf=round(t.conf, 3),
                               clase=int(cls), nombre_clase=nombres.get(cls, "desconocido") if cls >= 0 else "desconocido",
                               en_movimiento=bool(t.moving), desplazamiento_px=tuple(float(x) for x in t.vec / escala.k),
                               paralaje=bool(t.par)))
    return out


def como_array(salida):
    """TrackSalida -> Nx7 [x1, y1, x2, y2, id, conf, clase] (el formato que piden la mayoría de benchmarks MOT)."""
    return np.array([[*s.caja, s.id, s.conf, s.clase] for s in salida], np.float32).reshape(-1, 7)


class Cronometro:
    """Milisegundos de cada etapa del último frame (para saber si llega a tiempo real). tracker.tiempos_ms."""

    def __init__(self):
        self.ms, self._t = {}, None

    def inicio(self):
        self.ms, self._t = {}, time.perf_counter()

    def marca(self, etapa):
        t = time.perf_counter()
        self.ms[etapa] = round(1000 * (t - self._t), 2)
        self._t = t

### 4.3 Dibujo
**Verde** = quieto · **Rojo** = se mueve (con clase) · **Naranja** = se mueve (sin clase) · **Gris** = descartado por paralaje. El recuadro con número es el ID (cada ID, su color: un cambio de color sobre el mismo objeto = cambio de ID).

In [ ]:
# ======================================================================
# 4.3 · dibujar
# Qué hace : Dibuja la salida sobre el frame original (solo para ver; no afecta al tracking)
# ¿Se toca? : No
# ======================================================================
VERDE, ROJO, NARANJA, GRIS = (60, 200, 60), (40, 40, 255), (0, 150, 255), (170, 170, 170)


def color_id(tid):
    """Color estable por ID: un cambio de color sobre el mismo objeto = el tracker le cambió el ID."""
    c = cv2.cvtColor(np.uint8([[[int((tid * 47) % 180), 220, 230]]]), cv2.COLOR_HSV2BGR)[0, 0]
    return int(c[0]), int(c[1]), int(c[2])


def dibujar(frame, salida, titulo=None):
    """Verde = quieto · Rojo = se mueve (con clase) · Naranja = se mueve (sin clase) · Gris = descartado por paralaje.
    El recuadro con número es el ID (cada ID, su color)."""
    for s in salida:
        if not s.en_movimiento and s.clase < 0 and not s.paralaje:
            continue                                                     # propuesta quieta: no aporta nada
        color = GRIS if (s.paralaje and not s.en_movimiento) else (VERDE if not s.en_movimiento else (ROJO if s.clase >= 0 else NARANJA))
        x1, y1, x2, y2 = (int(round(v)) for v in s.caja)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        txt = f"{s.id}"
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(frame, (x1, max(0, y1 - th - 8)), (x1 + tw + 8, max(th + 8, y1)), color_id(s.id), -1)
        cv2.putText(frame, txt, (x1 + 4, max(th + 4, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, s.nombre_clase, (x1 + tw + 12, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    n_mov = sum(s.en_movimiento for s in salida)
    cv2.putText(frame, f"{titulo + ' | ' if titulo else ''}en movimiento: {n_mov} | tracks: {len(salida)}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    return frame

## PARTE 5 — Detector externo y bucle en tiempo real
### 5.1 Detector externo
**Entra:** un frame BGR original. **Sale:** `Nx6 [x1, y1, x2, y2, conf, clase]` en píxeles de **ese** frame, y `nombres = {id: "car", ...}`.

Aquí se simula con YOLO, con los mismos modelos y umbrales de la comparativa: OBB (DOTA) para tomas cenitales y COCO para oblicuas. **En el otro entorno solo se reemplaza esta celda.** El umbral de confianza del detector debe quedar **por debajo** de `conf_low` (0.10) o por lo menos no muy por encima: el tracker hace una 2ª ronda con las detecciones débiles.

In [ ]:
# ======================================================================
# 5.1 · Detector externo (simulado con YOLO)
# Qué hace : Devuelve las detecciones de un frame en el formato que espera el tracker
# ¿Se toca? : SÍ — en el otro entorno, esta celda se sustituye por el detector que ya existe
# ======================================================================
from ultralytics import YOLO

class DetectorYOLO:
    def __init__(self, pesos, clases, imgsz=1280, conf=0.15):
        self.modelo = YOLO(pesos)
        inv = {str(v).lower(): k for k, v in self.modelo.names.items()}
        self.clases = [inv[c] for c in clases]
        self.nombres = dict(self.modelo.names)
        self.imgsz, self.conf = imgsz, conf

    def __call__(self, frame):
        r = self.modelo.predict(frame, imgsz=self.imgsz, conf=self.conf, classes=self.clases, verbose=False)[0]
        b = r.obb if getattr(r, "obb", None) is not None else r.boxes     # OBB: se usa su caja horizontal
        if b is None or len(b) == 0:
            return np.zeros((0, 6), np.float32)
        return np.concatenate([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None], b.cls.cpu().numpy()[:, None]], 1).astype(np.float32)

DETECTOR_CENITAL = lambda: DetectorYOLO("../yolo11n-obb.pt", ("small vehicle", "large vehicle"), conf=0.15)
DETECTOR_OBLICUO = lambda: DetectorYOLO("../yolo11n.pt", ("car", "truck", "bus", "motorcycle"), conf=0.25)

### 5.2 Bucle frame a frame
Lee un frame, detecta, actualiza el tracker, dibuja y escribe: lo mismo que haría el otro entorno con la cámara en vivo.

Por cada frame guarda los ms del detector y los de cada etapa del tracker (`tracker.tiempos_ms`) y la cámara estimada (`tracker.calibrador`: pitch, altura y de dónde salen).

**Salidas** en `salidas/tiempo_real/`:
- `<video>_<tracker>.mp4`: video anotado.
- `<video>_<tracker>_tracks.csv`: tracks en formato MOT (`frame` empieza en 1), para el benchmark.

In [ ]:
# ======================================================================
# 5.2 · procesar
# Qué hace : Bucle frame a frame (detector -> tracker -> dibujo) con medición de tiempos
# ¿Se toca? : No (el tracker se crea con crear_tracker, en el Paso B)
# ======================================================================
def procesar(video, detector, crear_tracker, out_dir="salidas/tiempo_real", max_frames=None, out_width=1920):
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    tracker = crear_tracker(fps, detector.nombres)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    base = Path(out_dir) / f"{Path(video).stem}_{tracker.nombre}"
    writer, filas, mot = None, [], []
    k = 0
    while max_frames is None or k < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.perf_counter()
        dets = detector(frame)                                           # detector externo
        t_det = 1000 * (time.perf_counter() - t0)
        tracks = tracker.actualizar(frame, dets)                         # <- el tracker
        fila = dict(frame=k, detector_ms=round(t_det, 1), tracker_ms=round(sum(tracker.tiempos_ms.values()), 2),
                    **{f"{e}_ms": v for e, v in tracker.tiempos_ms.items()}, tracks=len(tracks),
                    en_movimiento=sum(t.en_movimiento for t in tracks))
        c = tracker.calibrador                                           # cámara estimada en este frame
        fila.update(pitch=round(c.pitch, 2), altura_m=round(c.altura, 1), fuente_pitch=c.fuente_pitch, fuente_altura=c.fuente_altura)
        filas.append(fila)
        mot += [[k + 1, *r] for r in como_array(tracks)]                 # formato MOT: frame, x1, y1, x2, y2, id, conf, clase
        vis = dibujar(frame.copy(), tracks, tracker.nombre)
        if out_width:
            vis = cv2.resize(vis, (out_width, int(round(vis.shape[0] * out_width / vis.shape[1]))), interpolation=cv2.INTER_AREA)
        if writer is None:
            writer = cv2.VideoWriter(f"{base}.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (vis.shape[1], vis.shape[0]))
        writer.write(vis)
        k += 1
    cap.release()
    if writer is not None:
        writer.release()
    pd.DataFrame(mot, columns=["frame", "x1", "y1", "x2", "y2", "id", "conf", "clase"]).to_csv(f"{base}_tracks.csv", index=False)
    df = pd.DataFrame(filas)
    print(f"{tracker.nombre}: {k} frames | tracker {df.tracker_ms.median():.1f} ms/frame (p95 {df.tracker_ms.quantile(0.95):.1f}) | "
          f"detector {df.detector_ms.median():.0f} ms/frame | IDs creados: {getattr(tracker, 'ids_emitidos', 'n/a')} | {base}.mp4")
    return df, tracker

## PARTE 6 — Ejecución
`PARTE 1` → `PARTE 5` solo definen cosas; aquí se ejecuta.

### Paso A — Pruebas (ejecutar primero, sobre todo en un entorno nuevo)
No necesitan video ni detector: comprueban cada pieza con datos sintéticos.

In [ ]:
# ======================================================================
# Paso A · Pruebas
# Qué hace : Mapeo al suelo, asignación, CMC y calibración en línea con una cámara sintética
# ¿Se toca? : No — se ejecuta en el Paso A
# ======================================================================
def selftest():
    cfg = ConfigUCMC(use_proposals=False)
    K, f = matriz_K(960, 540, 82.0)
    m = Mapa(K, 0.0, 100.0)                                              # cenital a 100 m
    xy, _, ok = m.uv2xy(np.array([[480.0, 270.0], [580.0, 270.0]]), np.tile(np.eye(2), (2, 1, 1)))
    assert ok.all() and np.allclose(xy[0], 0, atol=1e-6) and np.isclose(xy[1, 0], 100 * 100 / f), "mapeo cenital"
    assert np.allclose(m.xy2uv(xy), [[480, 270], [580, 270]]), "ida y vuelta"
    pares, ud, ut = asignar_con_limite(np.array([[1.0, 50.0], [60.0, 200.0]]), 100.0)
    assert pares == [(0, 0)] and ud == [1] and ut == [1], "asignación con límite"
    # objeto QUIETO en el suelo, paneo de 14 px/frame: con CMC conserva su ID
    trk = TrackerUCMC(30.0, CamaraUCMC(pitch_respaldo_deg=0.0, altura_m=100.0), cfg)
    trk.escala = type("E", (), {"ww": 960, "wh": 540})()
    trk.calibrador = CalibradorCamara(cfg, 30.0, trk.camara, 960, 540)
    for fr in range(60):
        x = 200 + 14 * fr
        Hf = np.array([[1, 0, 14 if fr else 0], [0, 1, 0], [0, 0, 1]], np.float64)
        trk.seguir(np.array([[x, 100, x + 12, 112, 0.9, 0]], np.float32), Hf, True); trk.frame += 1
    assert trk.ids_emitidos == 1, trk.ids_emitidos
    # calibración en línea: el gimbal baja de 50° a 20° mientras el dron avanza y baja de 80 a 60 m
    ww, wh, fps, N = 960, 540, 30.0, 300
    verdad, alt_v = np.linspace(50, 20, N), np.linspace(80, 60, N)
    maps = []
    for k in range(N):
        mk = Mapa(K, float(verdad[k]), float(alt_v[k]))
        th = np.deg2rad(verdad[k])
        R = np.array([[1.0, 0, 0], [0, -np.cos(th), -np.sin(th)], [0, np.sin(th), -np.cos(th)]])
        T = -R @ np.array([0.0, 0.25 * k, alt_v[k]])                     # avanza 7.5 m/s y baja
        mk.A = K @ np.column_stack([R[:, 0], R[:, 1], T]); mk.InvA = np.linalg.inv(mk.A)
        maps.append(mk)
    cal = CalibradorCamara(ConfigUCMC(), fps, CamaraUCMC(pitch_respaldo_deg=0.0), ww, wh)
    lado = lambda k: f * 4.5 / (alt_v[k] / np.cos(np.deg2rad(verdad[k])))   # un vehículo de 4.5 m en el centro
    err_p, err_a = [], []
    for k in range(N):
        H = np.eye(3) if k == 0 else maps[k].A @ maps[k - 1].InvA
        s = lado(k)
        dets = np.array([[480 - s / 2, 270 - s / 4, 480 + s / 2, 270 + s / 4, 0.9, 2]] * 3, np.float32)
        cal.actualizar(H, True, dets)
        if k > 2 * fps:                                                  # tras 2 s ya debe haber medido
            err_p.append(abs(cal.pitch - verdad[k])); err_a.append(abs(cal.altura / alt_v[k] - 1))
    # ser causal tiene un costo: la mediana de 5 muestras va ~1 s atrasada y entre muestras (0.5 s) se mantiene el
    # último valor. Aquí el gimbal gira 3°/s -> hasta ~4.5° de error mientras gira
    assert cal.fuente_pitch == "H" and max(err_p) < 5.0, f"pitch: error máx {max(err_p):.1f}°"
    assert max(err_a) < 0.08, f"altura: error máx {100 * max(err_a):.1f} %"
    print(f"tracker_ucmc OK | mapeo, asignación, CMC (1 ID con paneo) | calibración en línea tras 2 s: "
          f"pitch error máx {max(err_p):.1f}°, altura error máx {100 * max(err_a):.1f} %")


def prueba_comun():
    """Escala ida y vuelta, aviso de formato [x, y, w, h] y detecciones vacías."""
    esc = Escala(Config(), (2160, 3840, 3))                              # frame 4K -> trabajo 960 x 540
    assert (esc.ww, esc.wh) == (960, 540)
    d = np.array([[400, 400, 480, 440, 0.9, 2]], np.float32)
    assert np.allclose(esc.a_original(esc.a_trabajo(d)[0, :4]), d[0, :4]), "ida y vuelta"
    try:
        EntradaDetecciones.revisar(np.array([[400, 400, 80, 40, 0.9, 2]]))     # [x, y, w, h] por error
        raise AssertionError("debió avisar de cajas al revés")
    except ValueError:
        pass
    assert EntradaDetecciones.revisar(None).shape == (0, 6) and EntradaDetecciones.revisar([]).shape == (0, 6)
    assert como_array([]).shape == (0, 7)
    print("común OK | escala ida y vuelta, aviso de formato [x, y, w, h], detecciones vacías")


def prueba_pipeline():
    """Pipeline completo con un video sintético: un auto cruza sobre fondo con textura y la cámara hace paneo.
    Debe salir 1 ID, en movimiento, con la caja en píxeles originales."""
    rng = np.random.default_rng(0)
    fondo = cv2.GaussianBlur((rng.random((1200, 4000)) * 255).astype(np.uint8), (7, 7), 0)
    fondo = cv2.cvtColor(fondo, cv2.COLOR_GRAY2BGR)
    W, H = 1920, 1080
    trk = TrackerUCMC(30.0, nombres_clases={2: "car"})
    for f in range(90):
        cam_x = 4 * f                                                    # la cámara se desplaza 4 px/frame
        frame = fondo[60:60 + H, cam_x:cam_x + W].copy()
        x = 300 + 12 * f - cam_x                                         # el auto avanza 12 px/frame en el suelo
        cv2.rectangle(frame, (x, 500), (x + 80, 540), (0, 0, 255), -1)
        s = trk.actualizar(frame, np.array([[x, 500, x + 80, 540, 0.9, 2]], np.float32))
    assert len(s) == 1 and s[0].en_movimiento and s[0].nombre_clase == "car", s
    assert abs(s[0].caja[0] - x) < 6, (s[0].caja, x)
    print(f"pipeline OK | 1 ID tras 90 frames, en movimiento, caja en píxeles originales | ms por frame: {trk.tiempos_ms}")


prueba_comun()
selftest()
prueba_pipeline()

### Paso B — Procesar un video

In [ ]:
# ======================================================================
# Paso B · Procesar un video
# Qué hace : Crea el tracker y procesa el video frame a frame
# ¿Se toca? : SÍ — VIDEO_PATH, detector, CAMARA y crear_tracker. Para probar rápido: max_frames=300
# ======================================================================
VIDEO_PATH = Path("/Users/Alex/Documents/videos_VIGIA/vigia_8.mp4")     # <-- CAMBIAR por tu video
detector = DETECTOR_OBLICUO()                                          # oblicua: DETECTOR_OBLICUO() | cenital: DETECTOR_CENITAL()
CAMARA = CamaraUCMC(hfov_deg=82,                                       # DATO de la cámara (ficha técnica), no se estima
                    pitch_respaldo_deg=50,                              # a ojo: oblicua ~50 | cenital 0. Solo hasta la 1ª medida
                    altura_m=None,                                      # None = automática | número = fija (telemetría)
                    nota="pitch a ojo; solo hasta la 1ª medida")

def crear_tracker(fps, nombres):
    cfg = ConfigUCMC(work_width=960, use_proposals=True)                # benchmark MOT: use_proposals=False
    return TrackerUCMC(fps, CAMARA, cfg, nombres)                       # modo MANUAL: ConfigUCMC(ucmc_calibrar=False)

df, tracker = procesar(VIDEO_PATH, detector, crear_tracker, max_frames=None)
display(df.describe().T[["50%", "max"]].rename(columns={"50%": "mediana"}).round(2))

In [ ]:
display(Video(f"salidas/tiempo_real/{VIDEO_PATH.stem}_{tracker.nombre}.mp4", embed=False, width=1100))

### Paso C — La cámara que estimó en línea
- **Pitch:** hasta la primera medida es el de respaldo (gris); después sale de la `H` (azul).
- **Altura:** al principio es la de defecto (100 m); después la fijan los vehículos y la `H`.

Si existe la calibración del v13 (`comparativa_v13/<video>/calibracion_ucmc.json`, hecha **sobre el video completo, antes de procesar**), se dibuja encima para comparar.

In [ ]:
import json
import matplotlib.pyplot as plt

fps = cv2.VideoCapture(str(VIDEO_PATH)).get(cv2.CAP_PROP_FPS) or 30.0
t = df.frame / fps
fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
resp = df.fuente_pitch == "respaldo"
ax[0].plot(t, df.pitch, color="tab:blue", label="en línea (este tracker)")
ax[0].plot(t[resp], df.pitch[resp], ".", color="0.6", ms=2, label="respaldo")
ax[1].plot(t, df.altura_m, color="tab:orange", label="en línea (este tracker)")
v13 = Path("../Historial de desarrollo de trackers") / "comparativa_v13" / VIDEO_PATH.stem / "calibracion_ucmc.json"
if v13.exists():
    c = json.load(open(v13))
    n = min(len(c["pitch_serie"]), len(df))
    ax[0].plot(np.arange(n) / fps, c["pitch_serie"][:n], "--", color="k", lw=1, label="v13 (video completo, offline)")
    if c.get("altura_serie"):
        ax[1].plot(np.arange(n) / fps, c["altura_serie"][:n], "--", color="k", lw=1, label="v13 (video completo, offline)")
ax[0].set_ylabel("pitch (°)"); ax[1].set_ylabel("altura (m)"); ax[1].set_xlabel("s")
for a in ax:
    a.grid(alpha=0.3); a.legend(fontsize=8)
ax[0].set_title(f"{VIDEO_PATH.stem}: cámara estimada por UCMC (hfov {CAMARA.hfov_deg}° fijo)")
plt.show()

### Paso D — ¿Llega a tiempo real?
ms por etapa (mediana y p95) del tracker, **sin** contar el detector. A 30 fps hay 33 ms por frame **para todo**, detector incluido. Si no llega: `use_proposals=False` y/o un `work_width` menor.

In [ ]:
etapas = [c for c in df.columns if c.endswith("_ms")]
display(df[etapas].describe(percentiles=[0.5, 0.95]).T[["50%", "95%", "max"]].round(2))